In [ ]:
%run NB00_setup

In [ ]:
%run NB00b_gold_config

# NB03 – Quality checks (gate)
Runs all checks on the gold tables and saves the results in `gold_quality_results`.
If any blocking check fails it raises an exception, so the Notebook activity in the pipeline fails and nothing after it runs (KPIs).
Before that it shows a demo: the same checks on deliberately bad data (not saved in gold).

In [ ]:
import uuid

RUN_FAIL_DEMO = True            # set to False in automatic runs to save time
SENSOR_RANGE_BLOCKING = False   # True = out-of-range sensor values stop the run (False = WARN only)
RUN_ID = datetime.datetime.utcnow().strftime("%Y%m%d%H%M%S") + "-" + uuid.uuid4().hex[:6]
RES_SCHEMA = ("run_id string, run_ts timestamp, run_type string, check_name string, table_name string, "
              "failed_rows long, status string, detail string")

def run_checks(fact, trip=None, ep=None, drv=None, run_type="real", expect=None):
    R = []
    rid = RUN_ID if run_type == "real" else RUN_ID + "-demo"

    def add(name, tbl, failed, detail="", severity="blocking"):
        failed = int(failed or 0)
        status = "PASS" if failed == 0 else ("FAIL" if severity == "blocking" else "WARN")
        R.append((rid, datetime.datetime.utcnow(), run_type, name, tbl, failed, status, detail))

    def fk(df, tbl, col, dim, dimcol):
        d = F.broadcast(spark.table(dim).select(F.col(dimcol).alias("_dk")).distinct())
        failed = df.where(F.col(col).isNotNull()).join(d, F.col(col) == F.col("_dk"), "left_anti").count()
        add(f"fk_{col}", tbl, failed, f"rows whose {col} has no match in {dim}.{dimcol}")

    def null_keys(df, tbl, cols):
        row = df.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols]).first()
        per = {c: int(row[c] or 0) for c in cols}
        add("no_null_keys", tbl, sum(per.values()), str({k: v for k, v in per.items() if v}) or "")

    def dups(df, tbl, col):
        d = df.groupBy(col).count().where("count > 1").count()
        add("no_duplicate_rows", tbl, d, f"keys appearing more than once in {col}")

    # ---------- fact_telemetry ----------
    T = "gold_fact_telemetry"
    for col, dim, dcol in [("vehicle_key", "gold_dim_vehicle", "vehicle_key"),
                           ("driver_key", "gold_dim_driver", "driver_key"),
                           ("route_key", "gold_dim_route", "route_key"),
                           ("date_key", "gold_dim_date", "date_key"),
                           ("time_key", "gold_dim_time", "time_key"),
                           ("anomaly_type_key", "gold_dim_anomaly_type", "anomaly_type_key")]:
        fk(fact, T, col, dim, dcol)
    null_keys(fact, T, ["event_id", "vehicle_key", "driver_key", "route_key", "date_key", "time_key", "anomaly_type_key"])
    dups(fact, T, "event_id")

    m = fact.agg(
        F.sum(((F.col("distance_km") < 0) | (F.col("fuel_used_l") < 0) | (F.col("idle_minutes") < 0)
               | (F.col("interval_sec") < 0) | (F.col("interval_sec") > MAX_GAP_SEC)).cast("int")).alias("neg"),
        F.sum((F.col("cold_in_range_cnt") > F.col("cold_reading_cnt")).cast("int")).alias("cold_bad"),
        F.sum(((F.col("date_key") < 20000101) | (F.col("date_key") > 20991231)).cast("int")).alias("date_bad"),
        F.sum(((F.col("speed_kmh") < 0) | (F.col("speed_kmh") > 250)).cast("int")).alias("speed"),
        F.sum(((F.col("rpm") < 0) | (F.col("rpm") > 8000)).cast("int")).alias("rpm"),
        F.sum(((F.col("engine_temp_c") < -40) | (F.col("engine_temp_c") > 300)).cast("int")).alias("engine_temp"),
        F.sum(((F.col("fuel_rate_lph") < 0) | (F.col("fuel_rate_lph") > 200)).cast("int")).alias("fuel_rate"),
        F.sum(((F.col("latitude") < -90) | (F.col("latitude") > 90)).cast("int")).alias("lat"),
        F.sum(((F.col("longitude") < -180) | (F.col("longitude") > 180)).cast("int")).alias("lon"),
        F.sum(((F.col("cargo_temp_c") < -60) | (F.col("cargo_temp_c") > 60)).cast("int")).alias("cargo_temp")).first()
    add("range_gold_measures", T, (m["neg"] or 0) + (m["cold_bad"] or 0) + (m["date_bad"] or 0),
        f"negative/oversized measures={m['neg']}, in_range>total={m['cold_bad']}, bad date_key={m['date_bad']}")
    sensors = {k: int(m[k] or 0) for k in ["speed", "rpm", "engine_temp", "fuel_rate", "lat", "lon", "cargo_temp"]}
    add("range_sensor_values", T, sum(sensors.values()), str(sensors),
        severity="blocking" if SENSOR_RANGE_BLOCKING else "warning")

    # ---------- fact_trip ----------
    if trip is not None:
        T2 = "gold_fact_trip"
        for col, dim, dcol in [("vehicle_key", "gold_dim_vehicle", "vehicle_key"),
                               ("driver_key", "gold_dim_driver", "driver_key"),
                               ("route_key", "gold_dim_route", "route_key"),
                               ("start_date_key", "gold_dim_date", "date_key")]:
            fk(trip, T2, col, dim, dcol)
        null_keys(trip, T2, ["trip_id", "vehicle_key", "driver_key", "route_key", "start_date_key"])
        dups(trip, T2, "trip_id")
        bad = trip.where((F.col("distance_km") < 0) | (F.col("fuel_used_l") < 0) | (F.col("duration_minutes") < 0)
                         | (F.col("refrigerated_readings_in_range") > F.col("refrigerated_readings_total"))).count()
        add("range_gold_measures", T2, bad, "negative distance/fuel/duration or in_range > total")

    # ---------- fact_anomaly_episode ----------
    if ep is not None:
        T3 = "gold_fact_anomaly_episode"
        for col, dim, dcol in [("vehicle_key", "gold_dim_vehicle", "vehicle_key"),
                               ("driver_key", "gold_dim_driver", "driver_key"),
                               ("route_key", "gold_dim_route", "route_key"),
                               ("anomaly_type_key", "gold_dim_anomaly_type", "anomaly_type_key"),
                               ("start_date_key", "gold_dim_date", "date_key")]:
            fk(ep, T3, col, dim, dcol)
        null_keys(ep, T3, ["episode_nk", "vehicle_key", "driver_key", "route_key", "anomaly_type_key", "start_date_key"])
        dups(ep, T3, "episode_nk")
        bad = ep.where((F.col("duration_seconds") < 0) | (F.col("end_time") < F.col("start_time"))).count()
        add("range_gold_measures", T3, bad, "negative duration or end_time < start_time")

    # ---------- dim_driver ----------
    if drv is not None:
        bad = (drv.groupBy("driver_id").agg(F.sum(F.col("is_current").cast("int")).alias("c"))
                  .where("c != 1").count())
        add("one_current_row_per_driver", "gold_dim_driver", bad, "drivers with 0 or 2+ current rows")

    # ---------- row counts vs silver ----------
    if expect:
        for name, tbl, got, want in expect:
            add(name, tbl, abs(int(got) - int(want)), f"gold={got} silver={want}")
    return R

def save_results(R):
    df = spark.createDataFrame(R, RES_SCHEMA)
    df.write.mode("append").format("delta").saveAsTable("gold_quality_results")
    return df

In [ ]:
# ======================= demo: the same checks on deliberately bad data (not saved in gold) =======================
if RUN_FAIL_DEMO:
    good = spark.table("gold_fact_telemetry").limit(2000).cache()
    good.count()
    bad = (good
           .withColumn("vehicle_key", F.when(F.col("event_id").substr(1, 1) == "a", F.lit(999999)).otherwise(F.col("vehicle_key")))
           .withColumn("date_key", F.when(F.col("event_id").substr(1, 1) == "b", F.lit(None).cast("int")).otherwise(F.col("date_key")))
           .withColumn("speed_kmh", F.when(F.col("event_id").substr(1, 1) == "c", F.lit(-50.0)).otherwise(F.col("speed_kmh"))))
    bad = bad.unionByName(good.limit(50))                       # duplicate event_id values
    cur_drv = spark.table("gold_dim_driver").where("is_current")
    dup_cur = cur_drv.limit(3).cache(); dup_cur.count()
    bad_drv = spark.table("gold_dim_driver").unionByName(dup_cur)  # drivers with two current rows

    R_demo = run_checks(bad, drv=bad_drv, run_type="demo")
    d = save_results(R_demo)
    print("DEMO on bad data: you should see FAIL for fk, null keys, duplicates and one_current:")
    d.select("check_name", "table_name", "failed_rows", "status", "detail").show(50, truncate=False)
else:
    print("demo skipped")

In [ ]:
# ======================= the real gate =======================
t0 = time.time()
fact = spark.table("gold_fact_telemetry")
trip = spark.table("gold_fact_trip")
ep   = spark.table("gold_fact_anomaly_episode")
drv  = spark.table("gold_dim_driver")

expect = [
    ("row_count_matches_silver", "gold_fact_telemetry", fact.count(),
     spark.table("silver_telemetry").select("event_id").distinct().count()),   # silver has repeated event_id values
    ("row_count_matches_silver", "gold_fact_trip", trip.count(),
     spark.table("silver_telemetry").where("trip_id IS NOT NULL").select("trip_id").distinct().count()),
    ("row_count_matches_silver", "gold_fact_anomaly_episode", ep.count(), spark.table(SRC_EPISODES).count()),
]
R = run_checks(fact, trip, ep, drv, run_type="real", expect=expect)
out = save_results(R)
out.select("check_name", "table_name", "failed_rows", "status", "detail").show(100, truncate=False)
log_run("NB03 quality_checks", t0)

failed = [r for r in R if r[6] == "FAIL"]
warned = [r for r in R if r[6] == "WARN"]
print(f"PASS={sum(1 for r in R if r[6]=='PASS')}  WARN={len(warned)}  FAIL={len(failed)}")
if failed:
    raise Exception("QUALITY GATE FAILED: " + "; ".join(f"{r[4]}.{r[3]} ({r[5]} rows)" for r in failed))
print("QUALITY GATE PASSED ✔")